In [1]:

import numpy as np
from argparse import Namespace
from solver.kolmogorov_flow import KolmogorovFlow2d
from generate_data import legacy_solver, gen_data
from prepare_data import shuffle_data, test_data, get_slice
from visualize import *
from data_generation import *   
from preprocessing import *
from loss import *
from sb import model
from datasets.dataset import *
from torch.utils.data import DataLoader

In [2]:
# 코드 작성을 위해 임의로 만든 샘플 데이터임. 추후 시뮬 데이터로 교체 예정
high = np.load("./data_test/NS-Re1000_T255.npy", mmap_mode="r")
high_f_1d = np.concatenate([high] * 10, axis=0)[:,[0],:,:] 
high_f_2d = np.concatenate([high] * 10, axis=0)

low = np.load("./data_test/NS-Re1000_T255.npy", mmap_mode="r")
low_f_1d = np.concatenate([low] * 10, axis=0)[:,[0],:,:]
low_f_2d = np.concatenate([low] * 10, axis=0)

# Shape은 이렇게 고정할 예정
# 1d data는 C=1, H->T, W->X
# 2d data는 C->T, H->X, W->Y
# B, C, H, W = data.shape

In [3]:
mask_high_1d = make_sensor_mask(high_f_1d, n_sensors=13, sparse_ratio=None, seed=0)
mask_low_1d = make_sensor_mask(low_f_1d, n_sensors=13, sparse_ratio=None, seed=0)

mask_high_2d = make_sensor_mask(high_f_2d, n_sensors=int(256*256*0.1), sparse_ratio=None, seed=0)
mask_low_2d = make_sensor_mask(low_f_2d, n_sensors=int(256*256*0.1), sparse_ratio=None, seed=0)

In [4]:
# dataset_visualization(high_f_2d, mask_high_2d, batch_idx=0)
# dataset_visualization(low_f_1d, mask_low_1d, batch_idx=0)

In [5]:
ds_1d = PDEPairDataset(low_f_1d, high_f_1d, None, mask_high_1d)
dl_1d = DataLoader(ds_1d, batch_size=32, shuffle=True, drop_last=False)

ds_2d = PDEPairDataset(low_f_2d, high_f_2d, None, mask_high_2d)
dl_2d = DataLoader(ds_2d, batch_size=32, shuffle=True, drop_last=False)

In [7]:
dl_1d.dataset

In [ ]:
steps = 10
eps_train = 0.01
eps_test = 0.01
sb_object = model.SB_model_ver1(dl_2d, [0,1],N_train=100000, base_lr=1e-3,
                              steps=steps,eps_train=eps_train,eps_test=eps_test,patience=500,B=high_f_1d.shape[0],lambda_=1e-3,clamp=False,save=True, mask_list=[mask_high_1d, mask_low_1d])

In [ ]:
sb_object.train()

  0%|          | 0/100000 [00:00<?, ?it/s]